# Лабораторная работа 12. Матчасть DL
Задача: реализовать и обучить нейронную сеть, состоящую из 3 нейронов (2 слоя), предсказывать значения функции XOR. При выполнении лабораторной запрещается использовать фреймворки для глубокого обучения (как PyTorch, Tensorflow, Caffe, Theano и им подобные).

В первую очередь ознакомиться с этим материалом (https://towardsdatascience.com/implementing-the-xor-gate-using-backpropagation-in-neural-networks-c1f255b4f20d).

Что необходимо реализовать, используя знания и фрагменты кода из ссылки выше:  
1. Класс Neuron, имеющий вектор весов self._weigths
2. Два метода класса Neuron: forward(x), backward(x, loss) - реализующих прямой и обратный проход по нейронной сети. 
   Метод forward должен реализовывать логику работу нейрона: умножение входа на вес self._weigths, сложение и функцию активации сигмоиду. 
   Метод backward должен реализовывать взятие производной от сигмоиды и используя состояние нейрона обновить его веса.
3. Реализовать с помощью класса Neuron нейронную сеть с архитектурой из трёх нейронов, предложенную в статье:
<img src="images/LessonsII/Neuron.png" alt="Neuron" height=40% width=40%>


4. Для красоты обернуть в класс Model с методами forward и backward, реализующими правильное взаимодействие нейронов на прямом и обратном проходах.
5. Реализовать тренировочный цикл следующего вида:

```  python
цикл (обучающие данные):
	y = model.forward(x)
	err = loss(y, label)
	model.backward(x, err)
```

In [2]:
import numpy as np

In [52]:
class Neuron:
    def __init__(self, weights, learning_rate = 0.1):
        self._weights = weights
        self._out = 0
        self._is_output = False
        self._input = None
        self._d = 0
        self._learning_rate = learning_rate
        self._delta = 0

    def set_as_output(self):
        self._is_output = True

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    @staticmethod
    def sigmoid_derivative(x):
        return x * (1 - x)

    def forward(self, x):
        """Метод forward должен реализовывать логику работу нейрона: 
        умножение входа на вес self._weigths, сложение и функцию активации сигмоиду."""
        self._input = np.array(x)
        self._d = self._weights[0]
        for i in range(len(x)):
            self._d += self._weights[i+1] * x[i]
        self._out = self.sigmoid(self._d)
        return self._out

    def backward(self, x, loss, next_layer_delta=None, next_layer_weight=None):
        """Метод backward должен реализовывать взятие производной от сигмоиды 
        и используя состояние нейрона обновить его веса."""
        if self._is_output:
            self._delta = loss * self.sigmoid_derivative(self._d)

        else:
            self._delta = (
                self.sigmoid_derivative(self._d) * next_layer_delta * next_layer_weight
            )
            gradient = [self._delta * 1]
            for i in range(len(x)):
                grad = self._delta * self.sigmoid_derivative(self._d)
                gradient.append(grad)
            grad_array = np.array(gradient, dtype=float)
            self._weights = self._weights - self._learning_rate * grad_array

        return self._delta

In [ ]:
class Model:
    def __init__(self, learning_rate = 0.1):
        self._learning_rate = learning_rate
        self._loss = 0

        self.neuron1 = Neuron(
            weights=[
                np.random.randn() * 0.1,
                np.random.randn() * 0.1,
                np.random.randn() * 0.1,
            ],
            learning_rate=learning_rate,
        )

        self.neuron2 = Neuron(
            weights=[
                np.random.randn() * 0.1,
                np.random.randn() * 0.1,
                np.random.randn() * 0.1,
            ],
            learning_rate=learning_rate,
        )
        self.neuron3 = Neuron(
            weights=[
                np.random.randn() * 0.1,
                np.random.randn() * 0.1,
                np.random.randn() * 0.1,
            ],
            learning_rate=learning_rate,
        )
        self.neuron3.set_as_output()

    @staticmethod
    def binary_cross_entropy(pred, label):
        CE = -label * np.log(pred) - (1 - label) * np.log((1 - pred))
        return CE.mean()

    def forward(self, X, label):

        hidden1 = self.neuron1.forward(X)
        hidden2 = self.neuron2.forward(X)

        hid_out = np.array([hidden1, hidden2])
        out = self.neuron3.forward(hid_out)

        self._loss = self.binary_cross_entropy(out, label)

    def backward(self, X):
        loss = self._loss
        hid_out = np.array([self.neuron1._out, self.neuron2._out])
        delta_out = self.neuron3.backward(hid_out, loss)

        delta1 = self.neuron1.backward(X, loss, next_layer_delta = delta_out, next_layer_weight=self.neuron3._weights[1])
        delta2 = self.neuron2.backward(X, loss, 
            next_layer_delta=delta_out, next_layer_weight=self.neuron3._weights[2]
        )

    def get_weights(self):
        print(
            self.neuron1._weights[0], self.neuron1._weights[1], self.neuron1._weights[2]
        , sep = ' ')
        print(
            self.neuron2._weights[0],
            self.neuron2._weights[1],
            self.neuron2._weights[2],
            sep=" ",
        )
        print(
            self.neuron3._weights[0],
            self.neuron3._weights[1],
            self.neuron3._weights[2],
            sep=" ",
        )

    def predict(self, in1, in2):
        return self.forward(in1, in2)

    def train(self, X, y, epochs=100):
        losses = []

        for epoch in range(epochs):
            ep_loss = 0
            for input_data, label in zip(X, y):
                self.forward(input_data, label)
                ep_loss += abs(self._loss)
                self.backward(input_data)
            avg_loss = ep_loss/len(X)
            losses.append(avg_loss)
        return losses

In [66]:
inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
expected_output = np.array([[0], [1], [1], [0]])

nn = Model(learning_rate=0.5)
print("before train")
nn.get_weights()
losses = nn.train(inputs, expected_output)
print("after train")
nn.get_weights()

before train
-0.19086386974561598 0.043839542414580865 -0.06598299612103363
-0.15401096173728515 0.1064046494526547 -0.07871102967123124
-0.20705224859451904 0.009167819206540318 -0.0035535697879151627
0.5960955558345784
0.8006845369595104
0.8006376795679794
0.5960616761424126
0.5960946882209631
0.8006853247134733
0.8006384765278809
0.5960612517259183
0.5960938181742425
0.8006861139417044
0.8006392749988485
0.5960608273202238
0.5960929456870466
0.8006869046435132
0.8006400749803201
0.5960604029339589
0.5960920707520001
0.8006876968181628
0.8006408764716878
0.596059978575837
0.59609119336172
0.8006884904648707
0.8006416794722975
0.5960595542546585
0.5960903135088188
0.8006892855828084
0.8006424839814492
0.5960591299793075
0.5960894311859026
0.8006900821711012
0.8006432899983951
0.5960587057587569
0.5960885463855711
0.8006908802288268
0.8006440975223404
0.5960582816020658
0.5960876591004187
0.800691679755016
0.8006449065524428
0.5960578575183803
0.5960867693230361
0.8006924807486513
0.80